# xMIP plugin demo

Shows an xMIP-derived plugin recipe as small, inspectable Woodpecker fixes composed into one workflow.

In [ ]:
import numpy as np
import woodpecker_xmip_plugin  # noqa: F401 - imports plugin fixes for editable installs

import woodpecker
from woodpecker.testing import make_cmip6

Create a tiny CMIP6 dataset and add xMIP-style issues: `i`/`j` axes, `longitude`/`latitude` names, negative longitudes, centimeter vertical units, and GFDL-CM4 metadata.

In [ ]:
dataset = make_cmip6(
    overrides={"source_id": "GFDL-CM4", "experiment_id": "historical"},
    seed=7,
)
dataset = dataset.isel(time=slice(0, 2), lat=slice(0, 2), lon=slice(0, 3))
dataset = dataset.rename({"lat": "j", "lon": "i"})
dataset = dataset.expand_dims({"lev": [0.0, 100.0]})
dataset["lev"].attrs["units"] = "centimeters"

longitude = np.broadcast_to(np.array([-10.0, 0.0, 10.0]), (dataset.sizes["j"], 3))
latitude = np.broadcast_to(
    np.asarray(dataset["j"].values)[:, None],
    (dataset.sizes["j"], dataset.sizes["i"]),
)
dataset["longitude"] = (("j", "i"), longitude)
dataset["latitude"] = (("j", "i"), latitude)

dataset

In [ ]:
{
    "dims": dict(dataset.sizes),
    "data_vars": list(dataset.data_vars),
    "coords": list(dataset.coords),
    "lev_units": dataset["lev"].attrs.get("units"),
    "longitude_min": float(dataset["longitude"].min()),
    "branch_time_in_parent": dataset.attrs.get("branch_time_in_parent"),
}

Load the bundled `xmip.cmip6_preprocessing` recipe.

In [ ]:
recipe = woodpecker.recipe.get("xmip.cmip6_preprocessing")

recipe.model_dump()

Check and dry-run the recipe. Dry-run checks do not simulate earlier steps, so the first report only includes fixes visible on the original dataset.

In [ ]:
findings = woodpecker.recipe.check(dataset, recipe)

findings.fix_ids

In [ ]:
result = woodpecker.recipe.fix(dataset, recipe, dry_run=True)

result.stats, result.preview

Apply the recipe in memory and inspect the result.

In [ ]:
result = woodpecker.recipe.fix(dataset, recipe, dry_run=False)

result.stats

In [ ]:
{
    "dims": dict(dataset.sizes),
    "data_vars": list(dataset.data_vars),
    "coords": list(dataset.coords),
    "lev_values": dataset["lev"].values.tolist(),
    "lev_units": dataset["lev"].attrs.get("units"),
    "lon_min": float(dataset["lon"].min()),
    "branch_time_in_parent": dataset.attrs.get("branch_time_in_parent"),
}

In [ ]:
woodpecker.recipe.check(dataset, recipe)